# Session 13 Lab: Advanced Prompting
**Course:** Noob to AI Expert | **Track:** Expert

Build prompt chains, engineer system prompts, validate structured output, and implement dynamic few-shot selection.

In [ ]:
!pip install anthropic -q
print('✅ Ready')

In [ ]:
import anthropic, json, re, os
os.environ.setdefault('ANTHROPIC_API_KEY', 'your-key-here')
client = anthropic.Anthropic()

def call(prompt, system='', max_tokens=512):
    r = client.messages.create(
        model='claude-haiku-4-5-20251001', max_tokens=max_tokens,
        system=system, messages=[{'role': 'user', 'content': prompt}]
    )
    return r.content[0].text

print('✅ Client ready')

## Part 1: Prompt Chains

In [ ]:
def analyze_review_chain(review: str) -> dict:
    # Step 1: Extract pros/cons
    pros_cons = call(f'Extract pros and cons from this review as a JSON object with keys "pros" (list) and "cons" (list).\n\nReview: {review}')

    # Step 2: Write one-sentence summary
    summary = call(f'Write a one-sentence summary of this product review.\n\nReview: {review}')

    # Step 3: Score 1-10
    score_raw = call(f'Rate this review 1-10. Return ONLY the integer number.\n\nReview: {review}')
    try:
        score = int(re.search(r'\d+', score_raw).group())
        score = max(1, min(10, score))
    except:
        score = 5

    return {'pros_cons': pros_cons, 'summary': summary, 'score': score}

result = analyze_review_chain('Great noise cancellation and sound quality! Battery life is excellent. However, the ear cups feel a bit tight after 2 hours and the carrying case is flimsy.')
for k, v in result.items():
    print(f'{k}: {v}')

## Part 2: System Prompt Engineering

In [ ]:
SUPPORT_SYSTEM = """
You are a customer support agent for PythonTools Pro, a code editor.

Persona: Friendly, concise, technically accurate.
Constraints:
- Never discuss competitor products
- Never promise features not currently available
- Never access or assume user account data
- If out of scope, say: "I'll connect you with the right team."

Format: Use bullet points for steps. Keep under 100 words.
"""

test_cases = [
    'How do I enable dark mode?',
    'Does VS Code have better autocomplete?',  # competitor
    'Will you add AI code generation?',         # future feature
    'What is the meaning of life?',             # out of scope
]

for tc in test_cases:
    print(f'User: {tc}')
    print(f'Bot:  {call(tc, system=SUPPORT_SYSTEM, max_tokens=150)}')
    print()